In [ ]:
import sys
import os
import numpy as np
import pandas as pd

from sklearn.cross_decomposition import PLSCanonical


sys.path.append(os.path.join(os.getcwd(), '../../src/'))
from utils_worm import read_utils

sys.path.append(os.path.join(os.getcwd(), '../../../../'))
from utils import plot_utils
from utils import train_utils

In [ ]:
import importlib
importlib.reload(read_utils)
importlib.reload(plot_utils)
importlib.reload(train_utils)

In [ ]:
file_ids = list(range(0, 33+1))
# file_ids = [0,   # minrevs
# 			1,   # minswitch
# 			7,   # maxswitch
# 			11,  # median
# 			12,  # maxrevs
# 			17,  # mean
# 			19,  # paper
# 			21]  # clean
# file_ids = [19]
plot_id = 19

bhl_selection = ['REV', 'VENTRAL_TURN', 'DORSAL_TURN', 'PAUSE']

dims = 2

In [ ]:
path = os.path.join(read_utils.get_data_path(), 'exported_data_gcamp')

# read
(file_ids, filenames,
 traces,
 continuous_behaviors,
 discrete_states, eigenworms,
 kymographs) = read_utils.get_datas(path, file_ids)

# clear nan values
for i in range(len(file_ids)):
	mask = eigenworms[i].notna().all(axis=1)

	traces[i] = traces[i][mask]
	continuous_behaviors[i] = continuous_behaviors[i][mask]
	eigenworms[i] = eigenworms[i][mask]
	discrete_states[i] = discrete_states[i][mask]
	kymographs[i] = kymographs[i][mask]

# naive ID unknown neurons
for i in range(len(file_ids)):
	traces[i].columns = pd.Index([
		f'{file_ids[i]:02d}_{n_id:03d}' if empty or type(label) is not str else label for n_id, (label, empty) in enumerate(zip(traces[i].columns, traces[i].columns == ''))
	])

# preprocess
pass

In [ ]:
for idx in range(len(discrete_states)):
	discrete_states[idx]['FWD'] = np.logical_or(
		discrete_states[idx]['FWD'].values,
		discrete_states[idx]['SLOWING'].values
	).astype(np.intp)
	discrete_states[idx] = discrete_states[idx].drop('SLOWING', axis=1)

	discrete_states[idx]['UNKNOWN'] = np.logical_or(
		discrete_states[idx]['UNKNOWN'].values,
		discrete_states[idx]['TRACKING_FAILURE'].values
	).astype(np.intp)
	discrete_states[idx] = discrete_states[idx].drop('TRACKING_FAILURE', axis=1)

	discrete_states[idx] = discrete_states[idx].reindex(columns=['FWD', 'REV', 'VENTRAL_TURN', 'DORSAL_TURN', 'PAUSE', 'UNKNOWN'])

In [ ]:
plot_idx = file_ids.index(plot_id)

In [ ]:
(modes_manifold, modes_manifold_b,
 recos_manifold, recos_manifold_b,
 loads_plsc_man, loads_plsc_man_b,
 vars_expl_mans) = train_utils.get_manifolds_plsc(traces,
												  [ds[bhl_selection] for ds in discrete_states],
												  scale=False,
												  n_ls=dims)

In [ ]:
save = True

In [ ]:
plot_utils.plot_manifolds([f'{str(file_id)}: {filenames[file_ids.index(file_id)]}' for file_id in file_ids],
						  modes_manifold,
						  discrete_states,
						  [vx[:2] for vx in vars_expl_mans], None,
						  title='PLSC manifolds', top_space=3.5,
						  save_path=f'../../out/plots/manifolds/{dims}-dim-PLSC' if save else None)

In [ ]:
# plot_utils.plot_manifolds_3d([f'{str(file_id)}: {filenames[file_ids.index(file_id)]}' for file_id in file_ids],
# 							 modes_manifold, discrete_states,
# 							 [vx[:3] for vx in vars_expl_mans], None,
# 							 title='PLSC manifolds', top_space=3.5,
# 							 save_path=f'../../out/plots/manifolds/{dims}-dim-PLSC_3d' if save else None)

In [ ]:
# plot_utils.plot_manifolds_3d_sep([f'{str(file_id)}: {filenames[file_ids.index(file_id)]}' for file_id in file_ids],
# 								 modes_manifold, discrete_states,
# 								 [vx[:3] for vx in vars_expl_mans], None,
# 								 titles=['PLSC manifolds'] * len(file_ids), top_space=3.5, animate=True,
# 								 save_path=f'../../out/plots/manifolds/{dims}-dim-PLSC_3d' if save else None)

In [ ]:
for i, trace in enumerate(traces):
	print(np.mean(trace.values, axis=0))

In [ ]:
traces_residual = [pd.DataFrame(t.values - t_rec.values, columns=t.columns) for t, t_rec in zip(traces, recos_manifold)]

for i, trace in enumerate(traces_residual):
	print(np.mean(trace.values, axis=0))

# autostereogram

In [ ]:
# # Rotate modes_manifold by ﷿﷿﷿﷿﷿﷿5 degrees around the second axis (y-axis)
# # and 20 degrees around the first axis (x-axis)
# angle_y_deg = 10
# angle_x_deg = 20
# angle_y_rad = np.deg2rad(angle_y_deg)
# angle_x_rad = np.deg2rad(angle_x_deg)
#
# # Rotation matrix for x-axis (3D)
# R_x = np.array([
#     [1, 0, 0],
#     [0, np.cos(angle_x_rad), -np.sin(angle_x_rad)],
#     [0, np.sin(angle_x_rad), np.cos(angle_x_rad)]
# ])
#
# # Rotation matrices for y-axis (3D)
# R_y_left = np.array([
#     [np.cos(angle_y_rad), 0, np.sin(angle_y_rad)],
#     [0, 1, 0],
#     [-np.sin(angle_y_rad), 0, np.cos(angle_y_rad)]
# ])
# R_y_right = np.array([
#     [np.cos(-angle_y_rad), 0, np.sin(-angle_y_rad)],
#     [0, 1, 0],
#     [-np.sin(-angle_y_rad), 0, np.cos(-angle_y_rad)]
# ])
#
# # Combined rotation: first x, then y
# R_left = R_x @ R_y_left
# R_right = R_x @ R_y_right
#
# modes_manifold_l = [m @ R_left for m in modes_manifold]
# modes_manifold_r = [m @ R_right for m in modes_manifold]


In [ ]:
# plot_utils.plot_manifolds([f'{str(file_id)}: {filenames[file_ids.index(file_id)]}' for file_id in file_ids],
# 						  modes_manifold_l,
# 						  discrete_states,
# 						  [vx[:2] for vx in vars_expl_mans], None,
# 						  title='PLSC manifolds', top_space=3.5,
# 						  save_path=f'../../out/plots/manifolds/{dims}-dim-PLSC_autostereo_l' if save else None)

In [ ]:
# plot_utils.plot_manifolds([f'{str(file_id)}: {filenames[file_ids.index(file_id)]}' for file_id in file_ids],
# 						  modes_manifold_r,
# 						  discrete_states,
# 						  [vx[:2] for vx in vars_expl_mans], None,
# 						  title='PLSC manifolds', top_space=3.5,
# 						  save_path=f'../../out/plots/manifolds/{dims}-dim-PLSC_autostereo_r' if save else None)